In [8]:
import pandas as pd
import re

**1. EDA**

&emsp;a) Preprocessing

In [67]:
def write_sents(file):
    print('Removing comments..')
    with open(file, 'r', encoding='utf-8') as f:
        sentences = []
        prompt = ""
        id = ""
        time = ""
        for line in f:
            if line.startswith('# prompt:'):
                # format sentence for tokenizing
                prompt = line[9:].rstrip()
                continue

            elif line.startswith('# '):
                cols = line.split(' ')
                cols = [col for col in cols if col != '']
                id = cols[1][5:] # prefixed with user:
                time = cols[-1][5:].rstrip() # prefixed with time:

            elif line != '\n':
                cols = line.split(' ')
                cols = [col for col in cols if col != '']

                line = id + '\t' + '\t'.join(cols[:6]) + '\t' + time + '\t' + cols[-1]

                sentences.append(line)
        f.close()

    print('Writing sentences...')
    print(sentences[-1])
    file = f"{file.split('.')[0]}_readable_{file.split('.')[-1]}.txt"

    with open(file, 'w', encoding='utf-8') as f:
        for sentence in sentences:
            f.write(sentence)

In [68]:
write_sents('data/fr_en.slam.20190204.train')

Removing comments..
Writing sentences...
GIYQ1bA5	Pw8nO+in0202	homme	NOUN	Gender=Masc|Number=Sing|fPOS=NOUN++	ROOT	0	3	0



&emsp;b) Explore Data

In [69]:
df = pd.read_csv('data/fr_en_readable_train.txt', sep='\t', header=None, names=['ID', 'token_id', 'token', 'POS', 'Morpho-Syntactic_Features', 'Dependency-Relation', 'Dependancy-Head', 'time', 'p_recall'])
df.head(6)

,ID,token_id,token,POS,Morpho-Syntactic_Features,Dependency-Relation,Dependancy-Head,time,p_recall
0,YjS/mQOx,8XTyQUAl0101,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,det,2,14.0,0
1,YjS/mQOx,8XTyQUAl0102,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,ROOT,0,14.0,0
2,YjS/mQOx,8XTyQUAl0201,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,nsubj,4,14.0,0
3,YjS/mQOx,8XTyQUAl0202,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,cop,4,14.0,0
4,YjS/mQOx,8XTyQUAl0203,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,det,4,14.0,0
5,YjS/mQOx,8XTyQUAl0204,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,ROOT,0,14.0,0


In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 926646 entries, 0 to 926645
Data columns (total 9 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ID                         926646 non-null  str    
 1   token_id                   926646 non-null  str    
 2   token                      926646 non-null  str    
 3   POS                        926646 non-null  str    
 4   Morpho-Syntactic_Features  926646 non-null  str    
 5   Dependency-Relation        926646 non-null  str    
 6   Dependancy-Head            926646 non-null  int64  
 7   time                       868207 non-null  float64
 8   p_recall                   926646 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 63.6 MB


In [54]:
df.describe()

,Dependancy-Head,time,p_recall
count,926646.000000,868207.000000,926646.000000
mean,1.907721,29.003748,0.162266
std,1.718631,723.370040,0.368695
min,0.000000,0.000000,0.000000
25%,0.000000,5.000000,0.000000
50%,2.000000,10.000000,0.000000
75%,3.000000,17.000000,0.000000
max,15.000000,94821.000000,1.000000


In [71]:
# missing values and duplicates
df = df.drop_duplicates()
print(df.isnull().sum())
print(f"Number of duplicate rows: {df.duplicated().sum()}")

ID                               0
token_id                         0
token                            0
POS                              0
Morpho-Syntactic_Features        0
Dependency-Relation              0
Dependancy-Head                  0
time                         58439
p_recall                         0
dtype: int64
Number of duplicate rows: 0


In [73]:
# rows_missing_time = df[df['time'].isnull()]
# rows_missing_time.head(5)

# NsrjY0A = df[(
#     df['ID'] == 'Nsr+jY0A')
#     & (df['token'] == "I eat.")]
# NsrjY0A.head(100)
"""
Here I identified that some excercises have null values for time and with this in mind, I will be removing those rows. They do not serve 
a purpose as the point of this data is to analyze the relationship between recall time and recall accuracy, given various features.
"""
df = df.dropna(subset=['time'])


In [ ]:
# Questions
# 1. What is the trend of recall time to ratio of recall_accuracy?
# Where is time null?
# why is there such a large max time value? Is it an outlier? What is the distribution of time values?


